# Titanic + PyCaret
Run `python download_data.py` first (Python 3.10/3.11).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import pandas as pd
from config import DATA_PATH, TARGET, SEED
df = pd.read_csv(DATA_PATH)
df.head()

## Look at the data

In [ ]:
print(df.shape)
df.isna().mean().sort_values(ascending=False)

In [ ]:
df[TARGET].value_counts(normalize=True)

In [ ]:
df.groupby("sex")[TARGET].mean()

## AutoML step by step

In [ ]:
from pycaret.classification import *
setup(df, target=TARGET, session_id=SEED, train_size=0.8)

In [ ]:
best = compare_models(n_select=1, sort="AUC")

In [ ]:
tuned = tune_model(best, optimize="AUC")

In [ ]:
evaluate_model(tuned)

In [ ]:
final = finalize_model(tuned)
save_model(final, str(Path.cwd().parent / "models" / "pycaret_best"))

## Leakage test
Put `boat` and `body` back and see how the score jumps. A result that good should make you suspicious.

In [ ]:
from sklearn.datasets import fetch_openml
raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto").frame
raw[TARGET] = raw[TARGET].astype(int)
leaky = raw.drop(columns=["name", "ticket", "cabin", "home.dest"])
setup(leaky, target=TARGET, session_id=SEED, train_size=0.8)
compare_models(n_select=3, include=["lr", "rf", "dt"])